# COS 380: Lab 1 
## Building a Reusable Text Preprocessing Module

**Main engineering goal:** build one reusable file:

`COS380_NLP/nlp_toolkit/preprocessing.py`

Every function you implement in this lab belongs in that file. Use this notebook to load data, import your functions, run the provided tests, compare outputs, and answer the written questions.

**Do not define the required reusable functions only in this notebook.**

Task 5 also requires the helper `get_wordnet_pos(tag)` in `preprocessing.py`.


In [1]:
from pathlib import Path
import sys
def find_course_root(start=None):
    start = Path(start or Path.cwd()).resolve()
    for candidate in [start] + list(start.parents):
        if (candidate / "nlp_toolkit").exists() and (candidate / "course_tools").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate COS380_NLP. Make sure you are working inside the Studio 0 folder."
    )

COURSE_ROOT = find_course_root()

if str(COURSE_ROOT) not in sys.path:
    sys.path.insert(0, str(COURSE_ROOT))

print("COS 380 root:", COURSE_ROOT)
print("Reusable Lab 1 file:", COURSE_ROOT / "nlp_toolkit" / "preprocessing.py")

COS 380 root: /Users/grace/github/COS380_NLP
Reusable Lab 1 file: /Users/grace/github/COS380_NLP/nlp_toolkit/preprocessing.py


## NLTK Setup

More background:

- NLTK data: https://www.nltk.org/data.html
- WordNet: https://www.nltk.org/howto/wordnet.html

You normally need to download these resources only once for the Python environment you are using.

In [2]:
import nltk

nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger_eng")


[nltk_data] Downloading package stopwords to /Users/grace/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /Users/grace/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /Users/grace/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /Users/grace/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


True

## Task 1:  Inspect the Airline Tweets Corpus (2 points)

Load the data into a DataFrame named **`tweets`**. Later tasks use this same variable, so run the notebook in order.

In [3]:
import pandas as pd

DATA_PATH = COURSE_ROOT / "data" / "airline_tweets" / "AirlineTweets.csv"

tweets = pd.read_csv(DATA_PATH)

print("Shape:", tweets.shape)
print("Columns:", tweets.columns.tolist())

tweets.head()

Shape: (14640, 15)
Columns: ['tweet_id', 'airline_sentiment', 'airline_sentiment_confidence', 'negativereason', 'negativereason_confidence', 'airline', 'airline_sentiment_gold', 'name', 'negativereason_gold', 'retweet_count', 'text', 'tweet_coord', 'tweet_created', 'tweet_location', 'user_timezone']


,tweet_id,airline_sentiment,airline_sentiment_confidence,negativereason,negativereason_confidence,airline,airline_sentiment_gold,name,negativereason_gold,retweet_count,text,tweet_coord,tweet_created,tweet_location,user_timezone
0,570306133677760513,neutral,1.0000,NaN,NaN,Virgin America,NaN,cairdin,NaN,0,@VirginAmerica What @dhepburn said.,NaN,2015-02-24 11:35:52 -0800,NaN,Eastern Time (US & Canada)
1,570301130888122368,positive,0.3486,NaN,0.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica plus you've added commercials t...,NaN,2015-02-24 11:15:59 -0800,NaN,Pacific Time (US & Canada)
2,570301083672813571,neutral,0.6837,NaN,NaN,Virgin America,NaN,yvonnalynn,NaN,0,@VirginAmerica I didn't today... Must mean I n...,NaN,2015-02-24 11:15:48 -0800,Lets Play,Central Time (US & Canada)
3,570301031407624196,negative,1.0000,Bad Flight,0.7033,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica it's really aggressive to blast...,NaN,2015-02-24 11:15:36 -0800,NaN,Pacific Time (US & Canada)
4,570300817074462722,negative,1.0000,Can't Tell,1.0000,Virgin America,NaN,jnardino,NaN,0,@VirginAmerica and it's a really big bad thing...,NaN,2015-02-24 11:14:45 -0800,NaN,Pacific Time (US & Canada)


In [4]:
# Provided inspection code
print("Sentiment counts:")
display(tweets["airline_sentiment"].value_counts())

print("Airline counts:")
display(tweets["airline"].value_counts())

print("Sample tweets:")
display(
    tweets[["airline_sentiment", "airline", "text"]]
    .sample(n=min(12, len(tweets)), random_state=39)
)

Sentiment counts:


airline_sentiment
negative    9178
neutral     3099
positive    2363
Name: count, dtype: int64

Airline counts:


airline
United            3822
US Airways        2913
American          2759
Southwest         2420
Delta             2222
Virgin America     504
Name: count, dtype: int64

Sample tweets:


,airline_sentiment,airline,text
3022,negative,United,@united Stuck in. ORD because United can't fin...
10090,negative,US Airways,@USAirways Cancelled Flightled my flight then ...
2188,negative,United,@United - flight Cancelled Flightled. 4 hrs at...
3084,negative,United,"Thanks @united, great news that u won't refund..."
1070,negative,United,@united done. Want me to send a screen shot of...
10071,negative,US Airways,@USAirways just informed of three hour delay. ...
12252,negative,American,@AmericanAir hi we have lost and found solutio...
11164,positive,US Airways,"@USAirways #success made flight , please thank..."
8143,neutral,Delta,"@JetBlue's CEO #pilots among ardent fans, Wall..."
6283,positive,Southwest,@SouthwestAir Thank you thank you thank you


### Task 1 observations

Write a short response addressing:

1. Which text features look like noise?  
2. Which features might still contain sentiment information?
3. Give examples of mentions, URLs, hashtags, numbers, capitalization, punctuation, contractions, or informal language that you noticed.

**Your response:**  

Some text features that look like noise are "@" and "#", as well as urls and words that are all caps. Some of these features, like capitalization, might still contain sentiment information. Some examples of these are Tweet_id 570306133677760513 which contains a mention to a @VirginAmerica and @Dhepburn. Also, Tweet 2437  capitalizes GREAT and FLATTERING.

## Task 2: Build Basic Cleaning Functions (4 points)

Open:

`COURSE_ROOT / "nlp_toolkit" / "preprocessing.py"`

Implement **all six functions in that one file**:

- `lowercase(text)`
- `remove_urls(text)`
- `remove_mentions(text)`
- `handle_hashtags(text)`
- `remove_punctuation(text)`
- `normalize_whitespace(text)`

Possible imports in `preprocessing.py` include:

```python
import re
import string
```

Save `preprocessing.py` before running the next cell.

In [5]:
# Import the functions you created in nlp_toolkit/preprocessing.py.
# If you edit preprocessing.py after importing it, restart the kernel
# or reload the module before testing again.

from nlp_toolkit.preprocessing import (
    lowercase,
    remove_urls,
    remove_mentions,
    handle_hashtags,
    remove_punctuation,
    normalize_whitespace,
)

In [6]:
# PROVIDED TEST CASES - do not replace these with easier examples.

CLEANING_TESTS = [
    "@United My Flight is DELAYED!!! See https://example.com #NeverAgain",
    "Thanks   @AmericanAir   for the 3-hour delay...   #frustrated",
    "More info: https://t.co/abc123 -- Gate A12 changed AGAIN!!!",
]

for text in CLEANING_TESTS:
    print("\nORIGINAL:", text)
    print("lowercase:           ", lowercase(text))
    print("remove_urls:         ", remove_urls(text))
    print("remove_mentions:     ", remove_mentions(text))
    print("handle_hashtags:     ", handle_hashtags(text))
    print("remove_punctuation:  ", remove_punctuation(text))
    print("normalize_whitespace:", normalize_whitespace(text))


ORIGINAL: @United My Flight is DELAYED!!! See https://example.com #NeverAgain
lowercase:            @united my flight is delayed!!! see https://example.com #neveragain
remove_urls:          @United My Flight is DELAYED!!! See  #NeverAgain
remove_mentions:      United My Flight is DELAYED!!! See https://example.com #NeverAgain
handle_hashtags:      @United My Flight is DELAYED!!! See https://example.com Never Again
remove_punctuation:   United My Flight is DELAYED See httpsexamplecom NeverAgain
normalize_whitespace: @United My Flight is DELAYED!!! See https://example.com #NeverAgain

ORIGINAL: Thanks   @AmericanAir   for the 3-hour delay...   #frustrated
lowercase:            thanks   @americanair   for the 3-hour delay...   #frustrated
remove_urls:          Thanks   @AmericanAir   for the 3-hour delay...   #frustrated
remove_mentions:      Thanks AmericanAir for the 3-hour delay... #frustrated
handle_hashtags:      Thanks @AmericanAir for the 3-hour delay... frustrated
remove_punctuat

### Task 2 questions

- What did you decide to do with hashtags such as `#NeverAgain`?
- Why is that choice reasonable for sentiment analysis?
- Are numbers something your cleaning functions remove, preserve, or handle elsewhere? Explain.

**Your response:**  
When it comes to hashtags like `#NeverAgain`, I decided to parse the camel case and separate into spaces. It is reasonable to keep the body of the hashtag and split it into words. This is because the hashtag could still have a sentiment. The difficulty of this will be if there are multiple words in the same case. For example, my code would not handle `#neveragain` or `#NEVERAGAIN`. For numbers, I decided to preserve them without changes. Sometimes numbers are identifiers, like flight numbers, and sometimes they mean other things, such as "3-hour delay". I preserved them and didnt handle numbers because there were too many cases.

## Task 3: Build and Test a Tokenizer (4 points)

Implement `tokenize(text)` in the **same** `nlp_toolkit/preprocessing.py` file.

You may use Python string operations or regular expressions. Do not use a complete library preprocessing pipeline.

Save the file, then run the provided tests below.

In [7]:
from nlp_toolkit.preprocessing import tokenize

TOKENIZER_TEST_TWEETS = [
    "I can't believe Flight AA123 is delayed again!!!",
    "@United thanks for nothing... #NeverAgain",
    "Great crew, but my bag is still missing :(",
    "New York-based flight was 2 hrs late - why?",
    "Visit https://t.co/example for details; no help at all.",
]

for i, text in enumerate(TOKENIZER_TEST_TWEETS, start=1):
    print(f"{i}. {text}")
    print("   TOKENS:", tokenize(text))

1. I can't believe Flight AA123 is delayed again!!!
   TOKENS: ['I', "can't", 'believe', 'Flight', 'AA123', 'is', 'delayed', 'again!!!']
2. @United thanks for nothing... #NeverAgain
   TOKENS: ['@United', 'thanks', 'for', 'nothing...', '#NeverAgain']
3. Great crew, but my bag is still missing :(
   TOKENS: ['Great', 'crew,', 'but', 'my', 'bag', 'is', 'still', 'missing', ':(']
4. New York-based flight was 2 hrs late - why?
   TOKENS: ['New', 'York', 'based', 'flight', 'was', '2', 'hrs', 'late', '-', 'why?']
5. Visit https://t.co/example for details; no help at all.
   TOKENS: ['Visit', 'https://t.co/example', 'for', 'details;', 'no', 'help', 'at', 'all.']


### Task 3 analysis

Study the output produced by **your tokenizer**.

- How are contractions handled?
- What happens to hyphenated expressions?
- What happens to hashtags and mentions?
- How are numbers handled?
- Notice that one test contains a URL. Does your tokenizer itself handle the URL, or should `remove_urls()` be called before tokenization?
- What assumptions does your tokenizer make about what counts as a token?

There is not one required token list. Your explanation should accurately describe your implementation.

**Your response:**  
For my tokenizer, I replaced "-" with " " when the preceding and proceeding words were both letters; next I simply split on spaces. Contractions are seen as one word. Hyphenated expressions are taken as multiple words, split on the hyphen. Hashtags, mentions, numbers, and urls remain unchanged. In most cases, it is best to run the previous 6 preprocessing methods before the tokenizer, including `remove_urls()`. The tokenizer assumes that any group of characters not separated by spaces is a word. This creates interesting tokens such as ":(" which still carries meaning - this might be a case where we would lose something by running preprocessing to remove punctuation.

## Task 4: Compare Standard Stopwords with Negation-Preserving Stopwords (2 points)

Implement this function in the **same** `preprocessing.py` file:

`remove_stopwords(tokens, keep_negation=False)`

If you use NLTK:

```python
from nltk.corpus import stopwords
```

When `keep_negation=False`, use ordinary stopword removal.  
When `keep_negation=True`, preserve important negation terms such as `no`, `not`, and `never`.

In [8]:
from nlp_toolkit.preprocessing import remove_stopwords

NEGATION_TEST_SENTENCES = [
    "I do not like this airline",
    "No one helped me at the gate",
    "I will never fly with them again",
]

for text in NEGATION_TEST_SENTENCES:
    tokens = tokenize(text.lower())
    standard = remove_stopwords(tokens, keep_negation=False)
    preserve_negation = remove_stopwords(tokens, keep_negation=True)

    print("\nTEXT:", text)
    print("Original tokens:       ", tokens)
    print("Standard stopwords:    ", standard)
    print("Preserve negation:     ", preserve_negation)


TEXT: I do not like this airline
Original tokens:        ['i', 'do', 'not', 'like', 'this', 'airline']
Standard stopwords:     None
Preserve negation:      None

TEXT: No one helped me at the gate
Original tokens:        ['no', 'one', 'helped', 'me', 'at', 'the', 'gate']
Standard stopwords:     None
Preserve negation:      None

TEXT: I will never fly with them again
Original tokens:        ['i', 'will', 'never', 'fly', 'with', 'them', 'again']
Standard stopwords:     None
Preserve negation:      None


### Task 4 analysis

Compare the two outputs.

- What changed when `no`, `not`, and `never` were preserved?
- Could ordinary stopword removal change the apparent sentiment of one of these sentences?
- Which version would you prefer for airline sentiment analysis? Why?

**Your response:**  
TODO

## Task 5: Compare Stemming and Lemmatization (4 points)

Implement all three functions in the **same** `nlp_toolkit/preprocessing.py` file:

- `stem_tokens(tokens)`
- `get_wordnet_pos(tag)`
- `lemmatize_tokens(tokens)`

For this lab, lemmatization must use **part-of-speech information**.

Suggested imports:

```python
from nltk import pos_tag
from nltk.corpus import wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
```

Add this helper to `preprocessing.py`:

```python
def get_wordnet_pos(tag):
    """Convert NLTK POS tag to WordNet POS tag."""
    if tag.startswith("J"):
        return wordnet.ADJ
    elif tag.startswith("V"):
        return wordnet.VERB
    elif tag.startswith("N"):
        return wordnet.NOUN
    elif tag.startswith("R"):
        return wordnet.ADV
    else:
        return wordnet.NOUN
```

Your `lemmatize_tokens(tokens)` function should:

1. POS-tag the complete token list with `pos_tag(tokens)`.
2. Convert each NLTK tag using `get_wordnet_pos(tag)`.
3. Pass that POS value to `WordNetLemmatizer.lemmatize()`.

The setup cell downloads `wordnet`, `omw-1.4`, and `averaged_perceptron_tagger_eng`.


In [9]:
from nlp_toolkit.preprocessing import (
    stem_tokens,
    lemmatize_tokens,
)

# POS tagging is tested in sentence context.
MORPHOLOGY_TEST_SENTENCES = [
    "The passengers were running while the delayed flights were getting worse",
    "She studies the routes and is planning a better trip",
    "The crews are caring but the delays are getting longer",
]

for text in MORPHOLOGY_TEST_SENTENCES:
    tokens = tokenize(text.lower())
    print("\nTEXT:", text)
    print("Tokens:      ", tokens)
    print("Stemmed:     ", stem_tokens(tokens))
    print("Lemmatized:  ", lemmatize_tokens(tokens))



TEXT: The passengers were running while the delayed flights were getting worse
Tokens:       ['the', 'passengers', 'were', 'running', 'while', 'the', 'delayed', 'flights', 'were', 'getting', 'worse']
Stemmed:      None
Lemmatized:   None

TEXT: She studies the routes and is planning a better trip
Tokens:       ['she', 'studies', 'the', 'routes', 'and', 'is', 'planning', 'a', 'better', 'trip']
Stemmed:      None
Lemmatized:   None

TEXT: The crews are caring but the delays are getting longer
Tokens:       ['the', 'crews', 'are', 'caring', 'but', 'the', 'delays', 'are', 'getting', 'longer']
Stemmed:      None
Lemmatized:   None


### Task 5 analysis

Compare the stemmed and POS-aware lemmatized outputs.

- What differences do you notice?
- Find at least one word for which POS-aware lemmatization produces a more meaningful result than stemming.
- Why does supplying the POS tag help `WordNetLemmatizer`?
- Which method is more likely to produce strings that are not normal English words?
- For airline sentiment analysis, would you choose stemming, POS-aware lemmatization, or neither? Explain.

**Your response:**  
TODO


## Task 6: Compare Across Sentiment Categories (3 points)

There is **no required `preprocess()` function in Lab 1**.

Instead, this notebook provides a small helper that calls the reusable functions you already built. The helper is only for this lab analysis. The actual reusable work remains in `nlp_toolkit/preprocessing.py`.

The comparison below uses:

lowercase -> remove URLs -> remove mentions -> handle hashtags -> remove punctuation -> normalize whitespace -> tokenize -> negation-preserving stopword removal

In [10]:
# Notebook-only helper for Tasks 6 and 7.
# You are NOT required to move this helper into preprocessing.py.

def process_for_comparison(text):
    text = lowercase(text)
    text = remove_urls(text)
    text = remove_mentions(text)
    text = handle_hashtags(text)
    text = remove_punctuation(text)
    text = normalize_whitespace(text)
    tokens = tokenize(text)
    tokens = remove_stopwords(tokens, keep_negation=True)
    return tokens

In [11]:
assert "tweets" in globals(), (
    "The 'tweets' DataFrame is missing. Run Task 1 first, then continue in order."
)

samples = (
    tweets.groupby("airline_sentiment", group_keys=False)
          .sample(n=5, random_state=42)
          .reset_index(drop=True)
)

comparison = samples[["airline_sentiment", "text"]].copy()

comparison["processed_tokens"] = comparison["text"].map(
    process_for_comparison
)

display(comparison)

,airline_sentiment,text,processed_tokens
0,negative,@united gate C 24 IAD. U released passengers t...,None
1,negative,@USAirways 1729 connecting in charlotte to hou...,None
2,negative,@united installed and working are not the same...,None
3,negative,"@USAirways now I am on flight to FLL, and told...",None
4,negative,@USAirways ...Loosing a lot of business by usi...,None
5,neutral,JetBlue reading the NYTimes. “@JetBlue: Our fl...,None
6,neutral,@USAirways just realized my @AmericanAir advan...,None
7,neutral,@united I submitted a status match last week a...,None
8,neutral,@SouthwestAir flt 3260 out of mht. Have fun wi...,None
9,neutral,@united 4 open seats in 1st class on UA 2065. ...,None


In [ ]:
import textwrap

RANDOM_SEED = 11


def show_random_examples(df, n=10, seed=RANDOM_SEED, width=88):
    """Print a random sample of tweets with the full before/after text.

    Unlike display(df), nothing is truncated: long tweets are wrapped onto
    additional lines instead of being cut off with an ellipsis.
    Pass the same seed to get the same sample every run.
    """
    sample = df.sample(n=min(n, len(df)), random_state=seed)

    print(f"Random sample of {len(sample)} tweets (seed={seed})")

    for position, (index, row) in enumerate(sample.iterrows(), start=1):
        tokens = row["processed_tokens"]
        after = " ".join(tokens) if tokens else "(no tokens left)"

        print("\n" + "=" * width)
        print(f"[{position}] row {index}  |  sentiment: {row['airline_sentiment']}")
        print("-" * width)
        print("BEFORE:")
        print(textwrap.fill(row["text"], width=width,
                            initial_indent="    ", subsequent_indent="    "))
        print("AFTER:")
        print(textwrap.fill(after, width=width,
                            initial_indent="    ", subsequent_indent="    "))

    print("\n" + "=" * width)


show_random_examples(comparison, n=10)


### Task 6 analysis

From the displayed positive, neutral, and negative tweets:

- Identify one example where preprocessing improved consistency or removed distracting material.
- Identify one example where preprocessing removed something that might still be useful.
- Did the effect of preprocessing look the same across all sentiment categories?

**Your response:**  
TODO

## Task 7:  Reflection (1 point)

1. Which preprocessing decision required the most judgment from you?
2. Suppose the task changed from **sentiment analysis** to **airline identification**. Name two preprocessing choices you might reconsider and explain why.

**Your response:**  
TODO

## Final Self-Check

First import **all** required functions so they are visible to the checker.

In [12]:
from nlp_toolkit.preprocessing import (
    lowercase,
    remove_urls,
    remove_mentions,
    handle_hashtags,
    remove_punctuation,
    normalize_whitespace,
    tokenize,
    remove_stopwords,
    stem_tokens,
    get_wordnet_pos,
    lemmatize_tokens,
)

from course_tools.cos380_checks import validate_lab1
validate_lab1(globals())

COS 380 LAB 1 - VISIBLE SELF-CHECK
✓ lowercase found
✓ lowercase exists and is callable
✓ remove_urls found
✓ remove_urls exists and is callable
✓ remove_mentions found
✓ remove_mentions exists and is callable
✓ handle_hashtags found
✓ handle_hashtags exists and is callable
✓ remove_punctuation found
✓ remove_punctuation exists and is callable
✓ normalize_whitespace found
✓ normalize_whitespace exists and is callable
✓ tokenize found
✓ tokenize exists and is callable
✓ remove_stopwords found
✓ remove_stopwords exists and is callable
✓ stem_tokens found
✓ stem_tokens exists and is callable
✓ get_wordnet_pos found
✓ get_wordnet_pos exists and is callable
✓ lemmatize_tokens found
✓ lemmatize_tokens exists and is callable
✓ lowercase basic behavior
✓ remove_urls removes URL text
✓ remove_mentions removes @mention
✗ normalize_whitespace collapses repeated spaces
✓ tokenize returns a non-empty token sequence
✗ remove_stopwords supports keep_negation
    TypeError("'NoneType' object is not it

(15, 20)

## Before You Submit

Confirm that:

- [ ] All required functions are saved in `nlp_toolkit/preprocessing.py`.
- [ ] The notebook imports and actually uses those functions.
- [ ] You ran all provided test cases.
- [ ] You answered each analysis/reflection prompt.
- [ ] The notebook runs from top to bottom without errors.
- [ ] The final visible self-check runs successfully.